# 🌙 3LC Low-Light Detection Challenge — GPU runner

This notebook downloads the pipeline repo from GitHub and runs its scripts **one by one** on the GPU:

| Step | Script | What it does |
|---|---|---|
| 0 | `00_check_env.py` | PyTorch / CUDA / GPU check |
| 1 | `01_download_kit.py` | Starter kit (~625 MB) from the 3LC CDN, sha256-verified |
| 2 | `02_prepare_dataset.py` | Working dataset (images linked, labels copied) + `dataset.yaml` |
| 3 | `03_audit.py` | Class balance, darkness, label sanity |
| 4 | `04_train.py` | YOLOv11n @ 640 from the pinned official checkpoint, CUDA + AMP, DDP on 2 GPUs |
| 5 | `05_mine_labels.py` | Review queue of likely label errors + preview images |
| 6 | `06_apply_fixes.py` | Applies only the fixes you marked `accepted`, then retrain |
| 7 | `07_predict.py` | Test predictions → `submission.csv` |
| 8 | `08_validate_submission.py` | Format check (same rules as the metric) |
| 9 | `09_submit.py` | Optional upload via the Kaggle API |

**Before you run:** Settings → Accelerator → **GPU T4 x2** (or P100) · Settings → **Internet on** · join the competition and accept its rules.

## Settings

In [ ]:
# ---- edit these -------------------------------------------------------------
REPO_URL   = "https://github.com/<your-username>/lowlight-3lc.git"   # your fork/push of the repo
BRANCH     = "main"
RUN_NAME   = "baseline"     # first training run
EPOCHS     = 60             # ~1-1.5 min/epoch on T4 x2; use 2 for a quick smoke test
BATCH      = 32             # total across GPUs; must divide by the GPU count
MINE_LABELS = True          # step 5: build the review queue after training

# Step 6: path to a review_queue.csv where you've set status=accepted/rejected
# (download it from a previous run, edit, upload as a Kaggle Dataset). None = skip.
REVIEWED_QUEUE = None       # e.g. "/kaggle/input/my-reviewed-queue/review_queue.csv"
RETRAIN_NAME   = "v2_fixed"

TTA    = False              # test-time augmentation at predict; check the Rules first
SUBMIT = False              # True = upload via Kaggle API (needs a KAGGLE_API_TOKEN secret)
SUBMIT_MESSAGE = "yolo11n@640 lowlight"
# -----------------------------------------------------------------------------

## Download the repo

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "lowlight-3lc"
os.environ["LOWLIGHT_WORK"] = str(WORK / "lowlight")

if REPO.exists():
    shutil.rmtree(REPO)

# Fallback: the repo attached as a Kaggle Dataset (works with Internet off for the code).
attached = [p.parent for p in Path("/kaggle/input").glob("*/scripts/04_train.py")] + \
           [p.parent for p in Path("/kaggle/input").glob("*/*/scripts/04_train.py")]
if "<your-username>" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
elif attached:
    shutil.copytree(attached[0], REPO)
    print("Using repo attached as a dataset:", attached[0])
else:
    raise SystemExit("Set REPO_URL to your GitHub repo (or attach the repo folder as a Kaggle Dataset).")

os.chdir(REPO)
print("Repo at", REPO)
print(sorted(p.name for p in (REPO / "scripts").glob("*.py")))

A small helper that runs each script, streams its output live, and stops the notebook if a step fails.

In [ ]:
def run(*args):
    cmd = [sys.executable, "-u", *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"step failed (exit {p.returncode}): {' '.join(cmd[1:])}")

OUT = WORK                      # submission.csv, best_*.pt, audit/, review/ land here
RUNS = WORK / "lowlight" / "runs"

## Install dependencies
PyTorch with CUDA is already on Kaggle; this only adds Ultralytics and friends.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
import torch, ultralytics
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count(),
      "| ultralytics", ultralytics.__version__)

## Step 0 — environment check

In [ ]:
run("scripts/00_check_env.py")

## Step 1 — starter kit
Not on the Data tab: it's fetched from the 3LC CDN (needs Internet on). Re-runs resume and skip finished shards.

In [ ]:
run("scripts/01_download_kit.py")

## Step 2 — working dataset

In [ ]:
run("scripts/02_prepare_dataset.py")

## Step 3 — data audit

In [ ]:
run("scripts/03_audit.py")
from IPython.display import Image, display
if (OUT / "audit" / "class_balance.png").exists():
    display(Image(str(OUT / "audit" / "class_balance.png")))

## Step 4 — train (CUDA)
Uses every GPU it finds (DDP on T4 x2) with mixed precision. Hyperparameters live in `configs/train.yaml`; the model, checkpoint and image size are locked by the competition.

In [ ]:
run("scripts/04_train.py", "--name", RUN_NAME, "--epochs", EPOCHS, "--batch", BATCH)
for f in ["results.png", "confusion_matrix_normalized.png"]:
    p = RUNS / RUN_NAME / f
    if p.exists():
        display(Image(str(p), width=900))
print(open(OUT / f"provenance_{RUN_NAME}.json").read())

## Step 5 — mine likely label errors
Writes `review/review_queue.csv` and CLAHE-brightened previews (green = label, red = model). Download them, mark each row `accepted` / `rejected`, and feed the file back in Step 6.

In [ ]:
if MINE_LABELS:
    run("scripts/05_mine_labels.py", "--weights", RUNS / RUN_NAME / "weights" / "best.pt")
    import pandas as pd, glob
    q = pd.read_csv(OUT / "review" / "review_queue.csv")
    display(q["type"].value_counts())
    for f in sorted(glob.glob(str(OUT / "review" / "previews" / "*" / "*.jpg")))[:4]:
        print(f); display(Image(f, width=600))

## Step 6 — apply reviewed fixes and retrain (optional)
Runs only when `REVIEWED_QUEUE` points at a queue you've reviewed.

In [ ]:
FINAL_RUN = RUN_NAME
if REVIEWED_QUEUE:
    run("scripts/06_apply_fixes.py", "--queue", REVIEWED_QUEUE)            # dry run, prints the plan
    run("scripts/06_apply_fixes.py", "--queue", REVIEWED_QUEUE, "--apply")
    run("scripts/04_train.py", "--name", RETRAIN_NAME, "--epochs", EPOCHS, "--batch", BATCH)
    FINAL_RUN = RETRAIN_NAME
print("Predicting with run:", FINAL_RUN)

## Step 7 — predict the test set

In [ ]:
args = ["scripts/07_predict.py", "--weights", RUNS / FINAL_RUN / "weights" / "best.pt"]
if TTA:
    args.append("--tta")
run(*args)

## Step 8 — validate submission.csv

In [ ]:
run("scripts/08_validate_submission.py", OUT / "submission.csv")
import pandas as pd
pd.read_csv(OUT / "submission.csv").head()

## Step 9 — submit (optional)
Add a secret named `KAGGLE_API_TOKEN` (Add-ons → Secrets) or `KAGGLE_USERNAME` + `KAGGLE_KEY`. Otherwise download `/kaggle/working/submission.csv` from the Output panel and upload it on the Submit page.

In [ ]:
if SUBMIT:
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        for name in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
            try:
                os.environ[name] = s.get_secret(name)
            except Exception:
                pass
    except ImportError:
        pass
    run("scripts/09_submit.py", OUT / "submission.csv", "-m", SUBMIT_MESSAGE)
else:
    print("SUBMIT=False -> download /kaggle/working/submission.csv and upload it on the competition page.")

## Outputs

In [ ]:
for p in sorted(OUT.glob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.2f} MB  {p.name}")
# Zip the review queue + previews for download
if (OUT / "review").exists():
    shutil.make_archive(str(OUT / "review_bundle"), "zip", OUT / "review")
    print("review_bundle.zip ready")